In [1]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

c:\Users\koyel\Downloads\python_practice\ai_ml_learning_journey\nitish_singh\genai_notebooks\learning_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_core.documents import Document

# create Langchain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [3]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [4]:
vector_store = Chroma(
    embedding_function = OpenAIEmbeddings(), 
    persist_directory = 'chroma.db', 
    collection_name = 'sample'
)

- Converting docs to embeddings.
- .sqlite3 file will be created to use this chroma.db.
- In this stage, documents are not added to chromadb. just it got created.
- Added SqliteViewer extension to see chromadb inside vscode.

### Addition of Documents
- Unique ids will be assigned by default. 
- We can pass our own id as well.

In [5]:
vector_store.add_documents(docs) 

['452b7d86-9fa7-4c48-b9a5-91f9a4b5622a',
 '4e37737a-1d66-46dc-80cc-3c4a40fef48e',
 '4517c60d-c8c9-4c38-9c33-b1e7ec7aaeb4',
 '96d49226-c2c5-4c68-8738-04c339d58f6b',
 '870951fc-2305-4efa-a341-b57a5929fa56']

### View Documents

In [14]:
vector_store.get(include=['embeddings', 'documents', 'metadatas'])

{'ids': ['452b7d86-9fa7-4c48-b9a5-91f9a4b5622a',
  '4e37737a-1d66-46dc-80cc-3c4a40fef48e',
  '4517c60d-c8c9-4c38-9c33-b1e7ec7aaeb4',
  '96d49226-c2c5-4c68-8738-04c339d58f6b',
  '870951fc-2305-4efa-a341-b57a5929fa56'],
 'embeddings': array([[-0.00210453, -0.00214285,  0.0268    , ..., -0.01707893,
         -0.00366616,  0.01357884],
        [-0.00268021, -0.00010323,  0.02815653, ..., -0.01501936,
          0.00590092, -0.01164922],
        [ 0.00092799, -0.00476   ,  0.0124662 , ..., -0.01731381,
          0.00075886,  0.00296567],
        [-0.02714536,  0.00885395,  0.02699314, ..., -0.02592762,
          0.00900617, -0.01999116],
        [-0.01810451,  0.01281202,  0.0347942 , ..., -0.03034012,
         -0.00595078,  0.00521716]], shape=(5, 1536)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the m

### Search Documents
- Here documents similar to query will be fetchedcfrom the db.
- There is a parameter `k` which indicates number of similar objects to show in output.

First see for `k=2`.

In [ ]:
vector_store.similarity_search(
    query='who among these are a bowler?',
    k=2 
)

[Document(id='96d49226-c2c5-4c68-8738-04c339d58f6b', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='870951fc-2305-4efa-a341-b57a5929fa56', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.')]

Now see for `k=1`.

In [ ]:
vector_store.similarity_search(
    query='who among these are a bowler?',
    k=1
)

[Document(id='96d49226-c2c5-4c68-8738-04c339d58f6b', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

- Now search with similarity score among query and db documents.
- Will see similarity score as well; the lower the score, the more similar.

In [9]:
vector_store.similarity_search_with_score(
    query='who among these are a bowler?',
    k=2
)

[(Document(id='96d49226-c2c5-4c68-8738-04c339d58f6b', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.36824852228164673),
 (Document(id='870951fc-2305-4efa-a341-b57a5929fa56', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.4316979944705963)]

Now search with similarity score filtering based on metadata (here metadata is `team's name`).

In [10]:
vector_store.similarity_search_with_score(
    query = "",
    filter = {'team': 'Chennai Super Kings'} 
)

[(Document(id='4517c60d-c8c9-4c38-9c33-b1e7ec7aaeb4', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.6488258242607117),
 (Document(id='870951fc-2305-4efa-a341-b57a5929fa56', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.6566494703292847)]

### Update Documents
- Here we will generate a new docuemnt about Virat Kohli in earlier format.
- Next we will update that in db, so that earlier document will be replaced by the new one.

In [15]:
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

Below during updating db, as `document_id`, write id of document related to Virat kohli from above.

In [16]:
vector_store.update_document(
    document_id = '452b7d86-9fa7-4c48-b9a5-91f9a4b5622a', 
    document = updated_doc1
    ) 

View documents again to see document relatrd to Virat Kohli is updated.

In [ ]:
vector_store.get(include=['embeddings', 'documents', 'metadatas'])

{'ids': ['452b7d86-9fa7-4c48-b9a5-91f9a4b5622a',
  '4e37737a-1d66-46dc-80cc-3c4a40fef48e',
  '4517c60d-c8c9-4c38-9c33-b1e7ec7aaeb4',
  '96d49226-c2c5-4c68-8738-04c339d58f6b',
  '870951fc-2305-4efa-a341-b57a5929fa56'],
 'embeddings': array([[-0.00544442, -0.01907989,  0.00706373, ..., -0.01627786,
         -0.00032134,  0.00724619],
        [-0.00268021, -0.00010323,  0.02815653, ..., -0.01501936,
          0.00590092, -0.01164922],
        [ 0.00092799, -0.00476   ,  0.0124662 , ..., -0.01731381,
          0.00075886,  0.00296567],
        [-0.02714536,  0.00885395,  0.02699314, ..., -0.02592762,
          0.00900617, -0.01999116],
        [-0.01810451,  0.01281202,  0.0347942 , ..., -0.03034012,
         -0.00595078,  0.00521716]], shape=(5, 1536)),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple c

### Delete Documents
- Put id(s) of documents we want to delete.
- Here way of putting id(s) is always list, even for 1 id also.

Adding id of document realted to Virat Kohli.

In [18]:
vector_store.delete(ids=['452b7d86-9fa7-4c48-b9a5-91f9a4b5622a']) 

View documents to see deleted document is not present. As we deleted document related to Virat Kohli, it should not be visible.

In [19]:
vector_store.get(include=['embeddings', 'documents', 'metadatas'])

{'ids': ['4e37737a-1d66-46dc-80cc-3c4a40fef48e',
  '4517c60d-c8c9-4c38-9c33-b1e7ec7aaeb4',
  '96d49226-c2c5-4c68-8738-04c339d58f6b',
  '870951fc-2305-4efa-a341-b57a5929fa56'],
 'embeddings': array([[-0.00268021, -0.00010323,  0.02815653, ..., -0.01501936,
          0.00590092, -0.01164922],
        [ 0.00092799, -0.00476   ,  0.0124662 , ..., -0.01731381,
          0.00075886,  0.00296567],
        [-0.02714536,  0.00885395,  0.02699314, ..., -0.02592762,
          0.00900617, -0.01999116],
        [-0.01810451,  0.01281202,  0.0347942 , ..., -0.03034012,
         -0.00595078,  0.00521716]], shape=(4, 1536)),
 'documents': ["Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.',
  'Jasprit Bumrah i

- TODO:
    - Try with FAISS, Pinecone, Weaviate. Working is exactly same as chromadb.
    - Revisit video again to see vector store/db features.
    - Write what is cosine similarity, because this concept is used here.
    - Revisit that chromadb is shwoing just 1 row having valu (1,1) in two columns. Also post addion of documents can't visualize these documenst from db, though view and search commands are working normally. Take a look why data is not visible from db itself.  